# Normalization for Identity Resolution Phase: OurAirports

This notebook normalizes the OurAirports dataset (`OurAirports.xml`, shipped inside `schema/ourairports/OurAirports.zip`) for the identity resolution phase.

Steps applied:
1. Lowercase & remove punctuation
2. Strip spaces
3. Unicode normalization (replace ï, ö, ä, accented chars)
4. Translation *(SKIPPED – pending coaching session)*
5. Abbreviation expansion
6. Remove stopwords and common airport words
7. Country columns: restore `countryCode` for Namibia and add `countryName` from `countryCode` via pycountry

In [2]:
import re
import tempfile
import unicodedata
import zipfile
from pathlib import Path

import pandas as pd
import pycountry
from PyDI.io import load_xml

DATA_DIR = Path("../../schema/ourairports")
OUTPUT_DIR = Path("../../data/normalized_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ZIP_FILE = DATA_DIR / "OurAirports.zip"

# The mapped XML is only stored zipped: extract it to a temp dir to keep the repo clean
with tempfile.TemporaryDirectory() as tmp_dir:
    with zipfile.ZipFile(ZIP_FILE) as archive:
        xml_file = archive.extract("OurAirports.xml", tmp_dir)
    df = load_xml(xml_file, name="ourairports")
df.attrs['source'] = 'ourairports'

print(f"Loaded {len(df)} records")
print(f"Columns: {list(df.columns)}")
df.head(3)

Loaded 86151 records
Columns: ['id', 'name', 'type', 'scheduledService', 'localCode', 'lat', 'lon', 'elevation', 'countryCode', 'isoRegion', 'city', 'runway_lengthFt', 'runway_widthFt', 'runway_surface', 'runway_lighted', 'runway_closed', 'runway_leIdent', 'runway_heIdent', 'wikipedia', 'continent', 'iata', 'icao']


,id,name,type,scheduledService,localCode,lat,lon,elevation,countryCode,isoRegion,...,runway_widthFt,runway_surface,runway_lighted,runway_closed,runway_leIdent,runway_heIdent,wikipedia,continent,iata,icao
0,oa_1,Total RF Heliport,heliport,false,00A,40.070985,-74.933689,3.3528,US,US-PA,...,80,ASPH-G,false,false,H1,NaN,NaN,NaN,NaN,NaN
1,oa_2,Aero B Ranch Airport,small_airport,false,00AA,38.704022,-101.473911,1046.988,US,US-KS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,oa_3,Lowell Field,small_airport,false,00AK,59.947733,-151.692524,137.16,US,US-AK,...,40,GRVL,false,false,N,S,NaN,NaN,NaN,NaN


## 1. Lowercase & remove punctuation for string columns

Targeted columns: `name`, `city`

Removes: commas, dots, parentheses, `'`, `"`, hyphens (kept as space), and other punctuation.

Compared to the Kaggle notebook, the character sets are extended with punctuation that occurs in OurAirports names (`&`, `[`, `]`, `#`, `°`, curly quotes, en/em dashes, ...).

> Note: OurAirports has no `countryName` column — it will be added in step 7.

In [2]:
STRING_COLS = ["name", "city"]

# Characters to remove completely
REMOVE_CHARS = re.compile(r"[,\.\(\)'\"\!\?\:\;\`\~&\[\]#°’‘“”„´+*№]")
# Hyphens/slashes/dashes -> space so 'Saint-Pierre' becomes 'saint pierre'
HYPHEN_CHARS = re.compile(r"[-/–—]")

def lowercase_remove_punct(text):
    if not isinstance(text, str):
        return text
    text = text.lower()
    text = HYPHEN_CHARS.sub(" ", text)
    text = REMOVE_CHARS.sub("", text)
    return text

for col in STRING_COLS:
    df[col] = df[col].apply(lowercase_remove_punct)

print("Sample after step 1:")
df[["id", "name", "city"]].head(10)

Sample after step 1:


,id,name,city
0,oa_1,total rf heliport,bensalem
1,oa_2,aero b ranch airport,leoti
2,oa_3,lowell field,anchor point
3,oa_4,epps airpark,harvest
4,oa_5,katmai lodge airport,king salmon
5,oa_6,fulton airport,alex
6,oa_7,cordes airport,cordes
7,oa_8,goldstone gts airport,barstow
8,oa_9,williams ag airport,biggs
9,oa_10,kitchen creek helibase heliport,pine valley


## 2. Strip spaces

Remove leading/trailing whitespace and collapse multiple internal spaces.

In [3]:
MULTI_SPACE = re.compile(r"\s+")

def strip_spaces(text):
    if not isinstance(text, str):
        return text
    return MULTI_SPACE.sub(" ", text).strip()

for col in STRING_COLS:
    df[col] = df[col].apply(strip_spaces)

print("Sample after step 2:")
df[["id", "name", "city"]].head(10)

Sample after step 2:


,id,name,city
0,oa_1,total rf heliport,bensalem
1,oa_2,aero b ranch airport,leoti
2,oa_3,lowell field,anchor point
3,oa_4,epps airpark,harvest
4,oa_5,katmai lodge airport,king salmon
5,oa_6,fulton airport,alex
6,oa_7,cordes airport,cordes
7,oa_8,goldstone gts airport,barstow
8,oa_9,williams ag airport,biggs
9,oa_10,kitchen creek helibase heliport,pine valley


## 3. Unicode normalization

Uses NFKD decomposition to separate base characters from diacritics, then encodes to ASCII (ignoring the diacritic combining characters).

OurAirports has **thousands of non-English airport names** — this step is especially important here.

Examples:
- ï, ö, ä → i, o, a
- São Paulo → sao paulo
- Piñon → pinon
- Çarşamba → carsamba

Letters such as `ł`, `ø`, `ı` and `ß` have no NFKD decomposition and would be dropped silently (`wałęsa` → `waesa`), so they are mapped explicitly first.

Names written entirely in a non-Latin script become empty and are set to missing, because translation (step 4) is skipped.

In [4]:
# Latin letters that NFKD does not decompose (lowercase only: runs after step 1)
SPECIAL_LETTERS = str.maketrans({
    "ø": "o", "ı": "i", "ł": "l", "ß": "ss", "ð": "d",
    "æ": "ae", "đ": "d", "þ": "th", "œ": "oe", "ə": "e",
})

def normalize_unicode(text):
    if not isinstance(text, str):
        return text
    text = text.translate(SPECIAL_LETTERS)
    # NFKD decomposes combined chars; encode/decode drops combining diacritics
    return unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")

for col in STRING_COLS:
    # Strip again: dropping non-Latin characters can leave extra spaces behind
    df[col] = df[col].apply(normalize_unicode).apply(strip_spaces)
    empty = df[col] == ""
    print(f"{col}: {empty.sum()} values with no Latin characters set to missing")
    df.loc[empty, col] = pd.NA

# Verify: show entries that previously had diacritics
examples = ["são paulo", "piñon", "çarşamba", "düsseldorf", "gdańsk lech wałęsa", "borås", "kısıklı"]
for original in examples:
    print(f"  {original!r:25s} -> {normalize_unicode(original)!r}")

print("\nSample after step 3:")
df[["id", "name", "city"]].head(10)

name: 1 values with no Latin characters set to missing
city: 2 values with no Latin characters set to missing
  'são paulo'               -> 'sao paulo'
  'piñon'                   -> 'pinon'
  'çarşamba'                -> 'carsamba'
  'düsseldorf'              -> 'dusseldorf'
  'gdańsk lech wałęsa'      -> 'gdansk lech walesa'
  'borås'                   -> 'boras'
  'kısıklı'                 -> 'kisikli'

Sample after step 3:


,id,name,city
0,oa_1,total rf heliport,bensalem
1,oa_2,aero b ranch airport,leoti
2,oa_3,lowell field,anchor point
3,oa_4,epps airpark,harvest
4,oa_5,katmai lodge airport,king salmon
5,oa_6,fulton airport,alex
6,oa_7,cordes airport,cordes
7,oa_8,goldstone gts airport,barstow
8,oa_9,williams ag airport,biggs
9,oa_10,kitchen creek helibase heliport,pine valley


## 4. Translation *(SKIPPED FOR NOW: pending coaching session)*

OurAirports has thousands of airports with non-English names. Discuss with professor whether translation is worthwhile.

## 5. Abbreviation expansion

The OurAirports dataset uses far fewer abbreviations than Kaggle, but the map must stay aligned so that both datasets end up with the same long forms. The abbreviations below were identified by:
1. The known list from the task description
2. A frequency analysis of title-case and all-caps short tokens in the name column

Two Kaggle rules are left out because every occurrence in OurAirports means something else: `ab` (Swedish companies such as `Fjällflygarna AB`, codes such as `K9-AB-A`) and `pt` (Indonesian companies such as `PT Bumi Suksesindo`).

Expansion is applied as **whole-word** substitution (word boundaries `\b`) to avoid accidental replacements inside longer words.

In [5]:
# Abbreviation map (lowercase key -> expanded form, also lowercase)
# Same as the Kaggle notebook, without 'ab' and 'pt' (see above)
ABBREVIATION_MAP = [
    # Directly from task description
    (r"\bintl\b",  "international"),
    (r"\brgnl\b",  "regional"),
    (r"\bmuni\b",  "municipal"),
    (r"\bafb\b",   "air force base"),
    (r"\bfld\b",   "field"),
    # st: keep as 'saint' in airport context ("St. Louis" -> saint louis)
    # street is much less common in airport names than saint
    (r"\bst\.?\b", "saint"),
    # Additional abbreviations found via frequency analysis
    (r"\baaf\b",   "army air field"),
    (r"\bnas\b",   "naval air station"),
    (r"\bstn\b",   "station"),
    (r"\bco\.?\b", "county"),
    (r"\bmem\b",   "memorial"),
    (r"\blrrs\b",  "long range radar site"),
    (r"\bhwy\b",   "highway"),
    (r"\bmt\.?\b", "mount"),
    (r"\bft\.?\b", "fort"),
    (r"\bjct\.?\b","junction"),
    (r"\bblvd\.?\b","boulevard"),
]

# Compile patterns once
COMPILED_ABBREVS = [
    (re.compile(pattern, re.IGNORECASE), expansion)
    for pattern, expansion in ABBREVIATION_MAP
]

def expand_abbreviations(text):
    if not isinstance(text, str):
        return text
    for pattern, expansion in COMPILED_ABBREVS:
        text = pattern.sub(expansion, text)
    return text

# Only apply to 'name' column (abbreviations appear there, not in city)
name_before_abbrev = df["name"].copy()
df["name"] = df["name"].apply(expand_abbreviations)

# Strip spaces again after expansion (expansions may create multiple spaces)
df["name"] = df["name"].apply(strip_spaces)

changed = df["name"].notna() & (df["name"] != name_before_abbrev)
print(f"Abbreviation expansion changed {changed.sum()} records")
display(pd.DataFrame({"id": df["id"], "before": name_before_abbrev, "after": df["name"]})[changed].head(15))

Abbreviation expansion changed 571 records


,id,before,after
18,oa_19,st mary medical center heliport,saint mary medical center heliport
64,oa_65,st vincent general hospital heliport,saint vincent general hospital heliport
138,oa_139,st claire healthcare heliport,saint claire healthcare heliport
191,oa_192,osf st anthonys health center heliport,osf saint anthonys health center heliport
384,oa_385,st marys hospital heliport,saint marys hospital heliport
419,oa_420,st vincent jennings hospital heliport,saint vincent jennings hospital heliport
494,oa_495,st barnabas medical center heliport,saint barnabas medical center heliport
594,oa_595,rgnl medical center heliport,regional medical center heliport
602,oa_603,st josephs hospital heliport,saint josephs hospital heliport
650,oa_651,st francis hospital heliport,saint francis hospital heliport


## 6. Remove stopwords and common words

After abbreviation expansion many names still contain common words like `international`, `airport`, `the`, `of`, etc. that add noise to string comparators.

Strategy: remove English function words and Romance/Germanic articles from the `name` column.

In [6]:
# Common English function words (equivalent to sklearn's ENGLISH_STOP_WORDS subset)
# Defined inline to avoid importing sklearn/scipy (binary incompatibility on some macOS builds)
ENGLISH_STOP_WORDS = frozenset([
    "a", "an", "the", "and", "or", "but", "if", "in", "on", "at", "to",
    "of", "for", "by", "with", "as", "is", "it", "its", "be", "are", "was",
    "were", "been", "have", "has", "had", "do", "does", "did", "not", "no",
    "nor", "so", "yet", "both", "either", "neither", "than", "that", "this",
    "these", "those", "which", "who", "whom", "when", "where", "how", "all",
    "each", "every", "any", "some", "such", "from", "into", "through",
    "during", "before", "after", "above", "below", "between", "out", "off",
    "over", "under", "again", "further", "then", "once", "there", "here",
])

# Extended with Romance/Germanic articles & prepositions
STOPWORDS = ENGLISH_STOP_WORDS | {
    "de", "la", "le", "el", "al", "del", "des", "les", "du", "da", "di", "der", "das",
}

STOPWORD_PATTERN = re.compile(
    r"\b(" + "|".join(re.escape(w) for w in sorted(STOPWORDS, key=len, reverse=True)) + r")\b"
)

def remove_stopwords(text):
    if not isinstance(text, str):
        return text
    text = STOPWORD_PATTERN.sub("", text)
    return strip_spaces(text)

# Keep original name before stopword removal for inspection
df["name_before_sw"] = df["name"].copy()
df["name"] = df["name"].apply(remove_stopwords)

is_changed = df["name"].notna() & (df["name"] != df["name_before_sw"])
changed = df[is_changed][["id", "name_before_sw", "name"]].head(20)
print(f"Stopword removal changed {is_changed.sum()} records")
display(changed)

df.drop(columns=["name_before_sw"], inplace=True)

Stopword removal changed 6282 records


,id,name_before_sw,name
70,oa_71,the farm airport,farm airport
86,oa_87,flow through terminal heliport,flow terminal heliport
109,oa_110,la sal junction airport,sal junction airport
171,oa_172,the greenhouse airport,greenhouse airport
196,oa_197,lakeside lodge and marina seaplane base,lakeside lodge marina seaplane base
234,oa_235,del sol medical center heliport,sol medical center heliport
255,oa_256,la national guard heliport,national guard heliport
287,oa_288,field of dreams airport,field dreams airport
301,oa_302,rancho de aereo airport,rancho aereo airport
302,oa_303,o and g heliport,o g heliport


## 7. Country columns – restore `countryCode` and add `countryName` via pycountry

OurAirports has **only `countryCode`** (ISO alpha-2). Namibia's code `NA` was read as a missing value before the XML was created, so those records have no `countryCode`. `isoRegion` still contains it (`NA-KA`), so it is restored from there.

Then use pycountry exact lookup to add a canonical `countryName`, normalized with steps 1–3 like the other string columns:

```python
import pycountry
country = pycountry.countries.get(alpha_2="US")
# → Country(name='United States', ...)
```

`XK` (Kosovo) is not part of ISO 3166 and is added manually. `XP` and `ZZ` have no country and stay missing.

> Note: pycountry names can differ from the Kaggle names (`united states` vs `united states of america`), so **`countryCode` should be the blocking key**, not `countryName`.

In [7]:
# Restore Namibia's 'NA' code from isoRegion (e.g. 'NA-KA' -> 'NA')
missing_code = df["countryCode"].isna()
print(f"Restored countryCode for {missing_code.sum()} records:")
print(df.loc[missing_code, "isoRegion"].str.split("-").str[0].value_counts())
df.loc[missing_code, "countryCode"] = df.loc[missing_code, "isoRegion"].str.split("-").str[0]

# Codes that are not part of ISO 3166
EXTRA_COUNTRIES = {"XK": "Kosovo"}

def country_name_from_code(code):
    country = pycountry.countries.get(alpha_2=code)
    if country is None:
        return EXTRA_COUNTRIES.get(code)
    return country.name

def normalize_country_name(name):
    if name is None:
        return pd.NA
    return strip_spaces(normalize_unicode(strip_spaces(lowercase_remove_punct(name))))

# Look up each code once instead of once per record
country_names = {
    code: normalize_country_name(country_name_from_code(code))
    for code in df["countryCode"].unique()
}
df["countryName"] = df["countryCode"].map(country_names)

unmatched = df[df["countryName"].isna()]
print(f"\nNo countryName for {len(unmatched)} records:")
print(unmatched["countryCode"].value_counts())

print("\ncountryCode sample:")
print(df["countryCode"].value_counts().head(10))
df[["id", "countryCode", "countryName"]].drop_duplicates("countryCode").head(10)

Restored countryCode for 303 records:
isoRegion
NA    303
Name: count, dtype: int64

No countryName for 11 records:
countryCode
XP    9
ZZ    2
Name: count, dtype: int64

countryCode sample:
countryCode
US    32668
BR     8065
JP     3747
CA     3369
AU     2795
MX     2695
RU     1839
FR     1786
GB     1605
DE     1435
Name: count, dtype: int64


,id,countryCode,countryName
0,oa_1,US,united states
162,oa_163,PR,puerto rico
205,oa_206,MH,marshall islands
1012,oa_1013,MP,northern mariana islands
9351,oa_9352,GU,guam
9554,oa_9555,SO,somalia
9555,oa_9556,AQ,antarctica
9556,oa_9557,PG,papua new guinea
9557,oa_9558,CO,colombia
9558,oa_9559,AD,andorra


## Final: Save normalized dataset

> Note: Namibia's `countryCode` is the text `NA`, which pandas reads as missing by default. Read this file with `pd.read_csv(path, keep_default_na=False)`, otherwise those records lose their code again.

In [8]:
output_csv = OUTPUT_DIR / "ourairportsNormalized.csv"
df.to_csv(output_csv, index=False)
print(f"Saved {len(df)} normalized records to {output_csv}")

# Summary stats
print(f"\nDataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df[["id", "name", "countryCode", "countryName", "city"]].head(15)

Saved 86151 normalized records to ../../data/normalized_outputs/ourairportsNormalized.csv

Dataset shape: (86151, 23)
Columns: ['id', 'name', 'type', 'scheduledService', 'localCode', 'lat', 'lon', 'elevation', 'countryCode', 'isoRegion', 'city', 'runway_lengthFt', 'runway_widthFt', 'runway_surface', 'runway_lighted', 'runway_closed', 'runway_leIdent', 'runway_heIdent', 'wikipedia', 'continent', 'iata', 'icao', 'countryName']


,id,name,countryCode,countryName,city
0,oa_1,total rf heliport,US,united states,bensalem
1,oa_2,aero b ranch airport,US,united states,leoti
2,oa_3,lowell field,US,united states,anchor point
3,oa_4,epps airpark,US,united states,harvest
4,oa_5,katmai lodge airport,US,united states,king salmon
5,oa_6,fulton airport,US,united states,alex
6,oa_7,cordes airport,US,united states,cordes
7,oa_8,goldstone gts airport,US,united states,barstow
8,oa_9,williams ag airport,US,united states,biggs
9,oa_10,kitchen creek helibase heliport,US,united states,pine valley
